# 🩺 Diabetes Prediction — Complete Pipeline

| Step | Description |
|------|-------------|
| 1 | Install libraries |
| 2 | Download dataset from Kaggle |
| 3 | Load & explore data |
| 4 | Train & evaluate 4 models |
| 5 | Save best model |
| 6 | Download files for deployment |

> ▶️ **Runtime → Run all** to execute everything at once.

## 📦 Step 1 — Install Libraries

In [ ]:
!pip install kaggle imbalanced-learn scikit-learn pandas seaborn matplotlib --quiet
print('✅ Libraries installed!')

## 🔑 Step 2 — Download Dataset from Kaggle
> Get your API key from: kaggle.com → Account → API → Create New Token

In [ ]:
import os, json

KAGGLE_USERNAME = "your_username"   # ← change this
KAGGLE_KEY      = "your_api_key"    # ← change this

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d mathchi/diabetes-data-set --unzip -p /content/
print('✅ Dataset downloaded!')

## 🔬 Step 3 — Load & Explore Data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

data = pd.read_csv('/content/diabetes.csv')

print('Shape:', data.shape)
print('\nOutcome distribution:')
print(data['Outcome'].value_counts())
data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

data['Outcome'].value_counts().plot(kind='bar', ax=axes[0], color=['#4CAF50','#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Healthy (0)', 'Diabetic (1)'], rotation=0)

data.drop('Outcome', axis=1).hist(ax=axes[1], bins=15, color='steelblue')
axes[1].set_title('Feature Distributions')

plt.tight_layout()
plt.show()

## ⚙️ Step 4 — Train & Evaluate Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import RandomOverSampler
import numpy as np

X = data.drop('Outcome', axis=1)
y = data['Outcome']
FEATURE_NAMES = X.columns.tolist()

# Balance
ros = RandomOverSampler(random_state=41)
X_res, y_res = ros.fit_resample(X, y)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_res)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_res, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(),
    'SVC':                 SVC(probability=True),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42)
}

results = []
trained_models = {}
best_f1, best_model_name = 0, ''

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    results.append([name, round(acc,3), round(rec,3), round(prec,3), round(f1,3)])
    trained_models[name] = model

    if f1 > best_f1:
        best_f1, best_model_name = f1, name

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues',
                xticklabels=['Healthy','Diabetic'], yticklabels=['Healthy','Diabetic'])
    axes[i].set_title(f'{name}\nAccuracy: {acc:.2%} | F1: {f1:.2%}')
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
results_df = pd.DataFrame(results, columns=['Model','Accuracy','Recall','Precision','F1 Score'])
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print('Results sorted by F1 Score:')
print(results_df.to_string(index=False))
print(f'\n🏆 Best model: {best_model_name} (F1 = {best_f1:.2%})')

results_df.set_index('Model')[['Accuracy','Recall','Precision','F1 Score']].plot(
    kind='bar', figsize=(12, 5), colormap='viridis', width=0.7
)
plt.title('Model Comparison')
plt.xticks(rotation=20); plt.ylim(0, 1)
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 💾 Step 5 — Save Best Model

In [ ]:
import pickle

best_model = trained_models[best_model_name]

with open('best_model.pkl', 'wb') as f: pickle.dump(best_model, f)
with open('scaler.pkl',     'wb') as f: pickle.dump(scaler, f)
with open('features.pkl',   'wb') as f: pickle.dump(FEATURE_NAMES, f)

print(f'✅ Saved: {best_model_name}')
print('✅ Saved: scaler.pkl')
print('✅ Saved: features.pkl')

## 📥 Step 6 — Download Files for Deployment

In [ ]:
from google.colab import files

files.download('best_model.pkl')
files.download('scaler.pkl')
files.download('features.pkl')

print('✅ Downloaded! Now upload these 3 files to your GitHub repo alongside app.py and requirements.txt')